In [ ]:
import sys
# add parent directory to the path
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.autograd import Variable
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
import seaborn as sns
from tqdm import tqdm
from timm import create_model
import random
from facenet_pytorch import InceptionResnetV1
from models_vggface.vgg_face import vgg_face, VggFaceFeatureExtractor
import clip

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # Disable fast algorithms

device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')


In [2]:
class RareDiseaseDataset(Dataset):
    def __init__(self, df, image_dir, n_way=5, k_shot=1, q_query=1, num_tasks=1000, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.num_tasks = num_tasks
        self.transform = transform
        
        # Group by disease_name and filter
        self.disease_groups = df.groupby('disease_name').filter(
            lambda x: len(x) >= k_shot + q_query
        ).groupby('disease_name')
        self.diseases = list(self.disease_groups.groups.keys())
        # print("# of classes: ", len(self.diseases))
        
        if len(self.diseases) < n_way:
            raise ValueError(f"Not enough diseases ({len(self.diseases)}) for {n_way}-way task")
    
    def __len__(self):
        return self.num_tasks
    
    def __getitem__(self, idx):
        # seed_everything()
        sampled_diseases = torch.randperm(len(self.diseases))[:self.n_way]
        # print(sampled_diseases)
        sampled_disease_names = [self.diseases[i] for i in sampled_diseases]
        
        support_x = torch.zeros(self.n_way * self.k_shot, 3, 224, 224)
        support_y = torch.zeros(self.n_way * self.k_shot, dtype=torch.long)
        query_x = torch.zeros(self.n_way * self.q_query, 3, 224, 224)
        query_y = torch.zeros(self.n_way * self.q_query, dtype=torch.long)
        
        for i, disease in enumerate(sampled_disease_names):
            disease_df = self.disease_groups.get_group(disease)
            # seed_everything()
            sampled_indices = torch.randperm(len(disease_df))[:self.k_shot + self.q_query]
            # print(sampled_indices)
            
            for j, idx in enumerate(sampled_indices[:self.k_shot]):
                img_path = os.path.join(self.image_dir,
                                        disease_df.iloc[idx.item()]['disease_abbr'],
                                        disease_df.iloc[idx.item()]['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    support_x[i * self.k_shot + j] = self.transform(img)
                support_y[i * self.k_shot + j] = i
            
            for j, idx in enumerate(sampled_indices[self.k_shot:self.k_shot + self.q_query]):
                img_path = os.path.join(self.image_dir,
                                        disease_df.iloc[idx.item()]['disease_abbr'],
                                        disease_df.iloc[idx.item()]['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    query_x[i * self.q_query + j] = self.transform(img)
                query_y[i * self.q_query + j] = i
        
        return support_x, support_y, query_x, query_y
    

In [ ]:
class ProtoNet(nn.Module):
    def __init__(self, backbone='resnet152'):
        super(ProtoNet, self).__init__()
        self.backbone_name = backbone

        if backbone == 'resnet152':
            base_model = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
            self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
            self.output_dim = base_model.fc.in_features
        
        elif backbone == 'swin_t':
            base_model = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
            self.feature_extractor = nn.Sequential(
                base_model.features,
                nn.AdaptiveAvgPool2d(1)
            )
            self.output_dim = base_model.head.in_features
        
        elif backbone == 'densenet169':
            base_model = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
            self.feature_extractor = base_model.features
            self.output_dim = base_model.classifier.in_features
        
        elif backbone == 'inception_v3':
            base_model = models.inception_v3(weights='Inception_V3_Weights.DEFAULT')
            self.feature_extractor = nn.Sequential(
                base_model.Conv2d_1a_3x3,
                base_model.Conv2d_2a_3x3,
                base_model.Conv2d_2b_3x3,
                base_model.maxpool1,
                base_model.Conv2d_3b_1x1,
                base_model.Conv2d_4a_3x3,
                base_model.maxpool2,
                base_model.Mixed_5b,
                base_model.Mixed_5c,
                base_model.Mixed_5d,
                base_model.Mixed_6a,
                base_model.Mixed_6b,
                base_model.Mixed_6c,
                base_model.Mixed_6d,
                base_model.Mixed_6e,
                base_model.Mixed_7a,
                base_model.Mixed_7b,
                base_model.Mixed_7c,
                nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
            )
            self.output_dim = base_model.fc.in_features
            
        elif backbone == 'clip':
            self.model, _ = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")
            self.model.visual.float()  # Convert CLIP to FP32 to match the input
            self.feature_extractor = self.model.visual
            self.output_dim = 512  # CLIP's ViT-B/32 outputs 512-dimensional features

        # below are pretrained model with vggface
        elif backbone == 'facenet':
            self.feature_extractor = InceptionResnetV1(pretrained='vggface2', classify=False)
            self.output_dim = 512
        elif backbone == 'vggface':
            vggface = vgg_face('models_vggface/vgg_face.pth')
            self.feature_extractor = VggFaceFeatureExtractor(vggface_model=vggface, remove_last_conv=False)
            self.output_dim = 512
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
    def forward(self, x):
        x = self.feature_extractor(x)

        if self.backbone_name in ['clip', 'facenet']:
            x = x / x.norm(dim=-1, keepdim=True) if self.backbone_name == 'clip' else x
        else:
            x = F.adaptive_avg_pool2d(x, (1, 1)).view(x.size(0), -1)  # Global Average Pooling

        return x

In [4]:
def get_prototypical_logits(model, support_x, support_y, query_x):
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    embeddings_support = model(support_x)  # shape: (N_s, D)
    embeddings_query = model(query_x)      # shape: (N_q, D)

    classes = torch.unique(support_y)
    n_classes = len(classes)

    # Compute prototypes
    prototypes = torch.stack([
        embeddings_support[support_y == cls].mean(dim=0) for cls in classes
    ])  # shape: (n_classes, D)

    # Compute distances to prototypes
    dists = torch.cdist(embeddings_query, prototypes, p=2)  # (N_q, N_classes)
    logits = -dists  # more negative = further away → match CrossEntropyLoss

    return logits

In [5]:
def train_protonet(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for support_x, support_y, query_x, query_y in tqdm(dataloader, desc='Training', leave=True):
        support_x, support_y, query_x, query_y = (
            support_x.to(device), support_y.to(device),
            query_x.to(device), query_y.to(device)
        )

        # Flatten the input correctly
        support_x = support_x.view(-1, 3, 224, 224)
        query_x = query_x.view(-1, 3, 224, 224)

        support_y = support_y.view(-1)
        query_y = query_y.view(-1)
        
        logits = get_prototypical_logits(model, support_x, support_y, query_x)
        loss = criterion(logits, query_y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Correct loss and accuracy calculations
        total_loss += loss.item() * query_y.size(0)  # Multiply by batch size
        total_correct += (logits.argmax(dim=1) == query_y).sum().item()
        total_samples += query_y.size(0)

    # Print correct final values outside the loop
    epoch_loss = total_loss / total_samples  # Normalize loss
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

def evaluate_protonet(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for support_x, support_y, query_x, query_y in tqdm(dataloader, desc='Evaluating', leave=True):
            support_x, support_y, query_x, query_y = (
                support_x.to(device), support_y.to(device),
                query_x.to(device), query_y.to(device)
            )

            support_x = support_x.view(-1, 3, 224, 224)  
            query_x = query_x.view(-1, 3, 224, 224)  

            support_y = support_y.view(-1)
            query_y = query_y.view(-1)
            
            logits = get_prototypical_logits(model, support_x, support_y, query_x)
            loss = criterion(logits, query_y)

            total_loss += loss.item() * query_y.size(0)  # Accumulate weighted loss
            total_correct += (logits.argmax(dim=1) == query_y).sum().item()
            total_samples += query_y.size(0)

    # Compute final loss and accuracy
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy



In [ ]:
def prepare_kfold_dfs(model_name, way, shot, query, k_folds=5, seed=42):
    image_dir = "data/rd_images"
    csv_path = "data/disease_images.csv"
    data_df = pd.read_csv(csv_path)

    # Define transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    if model_name == 'clip':
        transform = clip.load('ViT-B/32')[1]

    # Group by disease and split
    disease_groups = data_df.groupby('disease_name')
    all_diseases = list(disease_groups.groups.keys())
    all_diseases = [disease for disease in all_diseases if len(disease_groups.get_group(disease)) > 1]

    # Step 1: Hold out 20% of diseases for test set
    seed_everything(seed)
    perm = torch.randperm(len(all_diseases)).tolist()
    num_total = len(all_diseases)
    num_test = int(num_total * 0.2)

    test_diseases = [all_diseases[i] for i in perm[:num_test]]
    cv_diseases = [all_diseases[i] for i in perm[num_test:]]

    test_df = data_df[data_df['disease_name'].isin(test_diseases)]
    test_dataset = RareDiseaseDataset(
        df=test_df,
        image_dir=image_dir,
        n_way=way,
        k_shot=shot,
        q_query=query,
        num_tasks=200,
        transform=transform
    )
    test_loader = DataLoader(test_dataset, batch_size=1, num_workers=4, shuffle=False)

    # Step 2: K-fold on remaining 80%
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    fold_loaders = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(cv_diseases)):
        train_diseases = [cv_diseases[i] for i in train_idx]
        val_diseases = [cv_diseases[i] for i in val_idx]
        # print(len(cv_diseases), len(train_diseases), len(val_diseases))

        train_df = data_df[data_df['disease_name'].isin(train_diseases)]
        val_df = data_df[data_df['disease_name'].isin(val_diseases)]

        train_dataset = RareDiseaseDataset(
            df=train_df,
            image_dir=image_dir,
            n_way=way,
            k_shot=shot,
            q_query=query,
            num_tasks=600,
            transform=transform
        )

        val_dataset = RareDiseaseDataset(
            df=val_df,
            image_dir=image_dir,
            n_way=way,
            k_shot=shot,
            q_query=query,
            num_tasks=100,
            transform=transform
        )

        train_loader = DataLoader(train_dataset, batch_size=1, num_workers=4, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=1, num_workers=4, shuffle=False)

        fold_loaders.append((train_loader, val_loader))

    return fold_loaders, test_loader
    

In [ ]:
def train_kfold(model_name, fold_loaders, test_loader, way=5, shot=1, query=1, device=None):
    num_epochs = 10
    all_fold_acc = []

    save_dir = f"logs_kfold_pn/{model_name}"
    os.makedirs(save_dir, exist_ok=True)

    for fold_idx, (train_loader, val_loader) in enumerate(fold_loaders):
        print(f"\n================= Fold {fold_idx + 1}/{len(fold_loaders)} =================")

        model = ProtoNet(backbone=model_name).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0
        acc_list = []

        fold_save_path = os.path.join(save_dir, f"fold{fold_idx + 1}_best_{way}way_{shot}shot.pth")

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            train_loss, train_acc = train_protonet(model, train_loader, optimizer, criterion, device)
            val_loss, val_acc = evaluate_protonet(model, val_loader, criterion, device)
            acc_list.append(val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), fold_save_path)
                print(f"New best model saved for fold {fold_idx + 1} with Val Acc: {best_acc:.4f}")

            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        all_fold_acc.append(best_acc)

    print(f"\n================= Cross-Validation Summary =================")
    print(f"Average Validation Accuracy across folds: {np.mean(all_fold_acc):.4f}")

def test_protonet(model_name, fold_id, test_loader, way, shot, query, device):
    print(f"\n===== Testing Best Model from Fold {fold_id} =====")

    # Initialize model
    model = ProtoNet(backbone=model_name).to(device)
    criterion = nn.CrossEntropyLoss()

    # Load saved weights from best model of this fold
    model_path = f"logs_kfold_pn/{model_name}/fold{fold_id}_best_{way}way_{shot}shot.pth"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")
    
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # Evaluate
    test_loss, test_acc = evaluate_protonet(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
    
    return test_loss, test_acc



In [8]:
# resnet152
model_name = 'resnet152'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: resnet152, Way: 5

================= Fold 1/5 =================



Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 41.46it/s]


New best model saved for fold 1 with Val Acc: 0.3400
Train Loss: 1.6243, Train Acc: 0.3097 | Val Loss: 1.8506, Val Acc: 0.3400

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 39.32it/s]


...

Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 6.1171, Val Acc: 0.2353

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:04<00:00, 20.73it/s]


Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 5.4096, Val Acc: 0.2200

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:04<00:00, 20.79it/s]

Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 6.6403, Val Acc: 0.2213

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.2061


In [ ]:
# resnet152 test
model_name = 'resnet152'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"\nModel: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
    
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")



Model: resnet152, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:10<00:00, 19.45it/s]


Test Loss: 2.1161 | Test Accuracy: 0.2120

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:10<00:00, 19.53it/s]


Test Loss: 1.7469 | Test Accuracy: 0.2230

===== Testing Best Model from Fold 3 =====


...

Test Loss: 4.3494 | Test Accuracy: 0.1717

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:09<00:00, 21.27it/s]


Test Loss: 4.6542 | Test Accuracy: 0.1953

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:09<00:00, 21.82it/s]

Test Loss: 4.8474 | Test Accuracy: 0.1710
Way 15 Average Test Accuracy: 17.99% ± 1.26% (1-sigma)


In [10]:
# densenet169
model_name = 'densenet169'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: densenet169, Way: 5

================= Fold 1/5 =================

Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:03<00:00, 33.11it/s]


New best model saved for fold 1 with Val Acc: 0.3180
Train Loss: 1.8291, Train Acc: 0.3877 | Val Loss: 1.7356, Val Acc: 0.3180

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:03<00:00, 32.15it/s]


Train Loss: 1.1215, Train Acc: 0.5473 | Val Loss: 2.1906, Val Acc: 0.2840

Epoch 3/10


...

Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 4.5417, Val Acc: 0.1993

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 19.76it/s]


Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 4.6514, Val Acc: 0.1873

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:04<00:00, 20.20it/s]

Train Loss: 0.0000, Train Acc: 1.0000 | Val Loss: 4.5307, Val Acc: 0.1747

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.1847


In [ ]:
# densenet169 test
model_name = 'densenet169'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"Model: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
        
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")


Model: densenet169, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:07<00:00, 26.04it/s]


Test Loss: 2.6821 | Test Accuracy: 0.2510

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:07<00:00, 25.53it/s]


Test Loss: 1.8665 | Test Accuracy: 0.2970

===== Testing Best Model from Fold 3 =====


...

Test Loss: 5.5457 | Test Accuracy: 0.1480

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:17<00:00, 11.26it/s]


Test Loss: 5.4441 | Test Accuracy: 0.2153

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:18<00:00, 10.68it/s]

Test Loss: 5.8543 | Test Accuracy: 0.1437
Way 15 Average Test Accuracy: 17.30% ± 3.45% (1-sigma)


In [14]:
# swin_t
model_name = 'swin_t'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: swin_t, Way: 5

================= Fold 1/5 =================

Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 44.37it/s]


New best model saved for fold 1 with Val Acc: 0.1980
Train Loss: 1.6484, Train Acc: 0.2080 | Val Loss: 1.6094, Val Acc: 0.1980

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 45.23it/s]


New best model saved for fold 1 with Val Acc: 0.2100
Train Loss: 1.6266, Train Acc: 0.2000 | Val Loss: 1.6094, Val Acc: 0.2100

Epoch 3/10


...

Train Loss: 2.7132, Train Acc: 0.0662 | Val Loss: 2.7080, Val Acc: 0.1060

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:13<00:00,  7.31it/s]


Train Loss: 2.7121, Train Acc: 0.0686 | Val Loss: 2.7080, Val Acc: 0.1167

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:10<00:00,  9.93it/s]

Train Loss: 5.5856, Train Acc: 0.0694 | Val Loss: 2.7080, Val Acc: 0.0807

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.1249


In [ ]:
# swin_t test
model_name = 'swin_t'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"Model: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")


Model: swin_t, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:06<00:00, 30.16it/s]


Test Loss: 1.6094 | Test Accuracy: 0.2140

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:07<00:00, 27.37it/s]


Test Loss: 1.6094 | Test Accuracy: 0.2260

===== Testing Best Model from Fold 3 =====


...

Test Loss: 2.7080 | Test Accuracy: 0.1207

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:16<00:00, 12.08it/s]


Test Loss: 2.7081 | Test Accuracy: 0.0690

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:16<00:00, 12.37it/s]

Test Loss: 2.7080 | Test Accuracy: 0.0660
Way 15 Average Test Accuracy: 8.03% ± 2.41% (1-sigma)


In [16]:
# clip
model_name = 'clip'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: clip, Way: 5

================= Fold 1/5 =================

Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:04<00:00, 22.98it/s]


New best model saved for fold 1 with Val Acc: 0.2500
Train Loss: 1.5943, Train Acc: 0.2770 | Val Loss: 1.5787, Val Acc: 0.2500

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 17.12it/s]


New best model saved for fold 1 with Val Acc: 0.2600
Train Loss: 1.5948, Train Acc: 0.2683 | Val Loss: 1.5930, Val Acc: 0.2600

Epoch 3/10


...

Train Loss: 2.7000, Train Acc: 0.0908 | Val Loss: 2.6883, Val Acc: 0.0780

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 17.69it/s]


Train Loss: 2.7022, Train Acc: 0.0877 | Val Loss: 2.7111, Val Acc: 0.0687

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 17.50it/s]

Train Loss: 2.7068, Train Acc: 0.0794 | Val Loss: 2.7091, Val Acc: 0.0880

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.1300


In [ ]:
# clip test
model_name = 'clip'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"Model: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")


Model: clip, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:07<00:00, 26.47it/s]


Test Loss: 1.6087 | Test Accuracy: 0.2710

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:06<00:00, 30.38it/s]


Test Loss: 1.6655 | Test Accuracy: 0.1730

===== Testing Best Model from Fold 3 =====


...

Test Loss: 2.7276 | Test Accuracy: 0.0660

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:17<00:00, 11.51it/s]


Test Loss: 2.7204 | Test Accuracy: 0.0567

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:21<00:00,  9.48it/s]

Test Loss: 2.7227 | Test Accuracy: 0.0970
Way 15 Average Test Accuracy: 7.30% ± 1.90% (1-sigma)


In [18]:
# facenet
model_name = 'facenet'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: facenet, Way: 5

================= Fold 1/5 =================

Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 36.77it/s]


New best model saved for fold 1 with Val Acc: 0.2460
Train Loss: 1.5975, Train Acc: 0.2543 | Val Loss: 1.5865, Val Acc: 0.2460

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 36.56it/s]


New best model saved for fold 1 with Val Acc: 0.3700
Train Loss: 1.5703, Train Acc: 0.2787 | Val Loss: 1.5588, Val Acc: 0.3700

Epoch 3/10


...

Train Loss: 1.6447, Train Acc: 0.9723 | Val Loss: 2.6924, Val Acc: 0.1260

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 19.97it/s]


Train Loss: 1.6382, Train Acc: 0.9809 | Val Loss: 2.7170, Val Acc: 0.1047

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 19.23it/s]

Train Loss: 1.8334, Train Acc: 0.7478 | Val Loss: 2.7247, Val Acc: 0.1160

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.1280


In [ ]:
# facenet test
model_name = 'facenet'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"Model: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")


Model: facenet, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:09<00:00, 20.22it/s]


Test Loss: 1.6309 | Test Accuracy: 0.2860

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:13<00:00, 15.18it/s]


Test Loss: 1.7623 | Test Accuracy: 0.2190

===== Testing Best Model from Fold 3 =====


...

Test Loss: 2.7336 | Test Accuracy: 0.0617

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:14<00:00, 13.92it/s]


Test Loss: 2.8356 | Test Accuracy: 0.0743

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:14<00:00, 13.82it/s]

Test Loss: 2.7148 | Test Accuracy: 0.1000
Way 15 Average Test Accuracy: 8.03% ± 1.40% (1-sigma)


In [20]:
# vggface
model_name = 'vggface'
for way in [5, 10, 15]:
    print(f"Model: {model_name}, Way: {way}")
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=1, query=1, device=device)

Model: vggface, Way: 5

================= Fold 1/5 =================

Epoch 1/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 49.14it/s]


New best model saved for fold 1 with Val Acc: 0.1540
Train Loss: 1.6527, Train Acc: 0.2153 | Val Loss: 1.6125, Val Acc: 0.1540

Epoch 2/10


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 46.03it/s]


New best model saved for fold 1 with Val Acc: 0.1880
Train Loss: 1.6155, Train Acc: 0.2157 | Val Loss: 1.6094, Val Acc: 0.1880

Epoch 3/10


...

Train Loss: 2.7081, Train Acc: 0.0667 | Val Loss: 2.7081, Val Acc: 0.0667

Epoch 9/10


Evaluating: 100%|██████████| 100/100 [00:10<00:00,  9.43it/s]


Train Loss: 2.7081, Train Acc: 0.0667 | Val Loss: 2.7081, Val Acc: 0.0667

Epoch 10/10


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 19.52it/s]


Train Loss: 2.7081, Train Acc: 0.0667 | Val Loss: 2.7081, Val Acc: 0.0667

================= Cross-Validation Summary =================
Average Validation Accuracy across folds: 0.0949


In [ ]:
# vggface test
model_name = 'vggface'
for way in [5, 10, 15]:
    test_acc_all = []
    print(f"Model: {model_name}, Way: {way}")
    _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=1, query=1)
    for i in range(5):
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=1, query=1, device=device)
        test_acc_all.append(test_acc)
    mean_acc = np.mean(test_acc_all)
    std_acc = np.std(test_acc_all, ddof=1)

    print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")


Model: vggface, Way: 5

===== Testing Best Model from Fold 1 =====


Evaluating: 100%|██████████| 200/200 [00:06<00:00, 30.39it/s]


Test Loss: 1.6094 | Test Accuracy: 0.1860

===== Testing Best Model from Fold 2 =====


Evaluating: 100%|██████████| 200/200 [00:05<00:00, 35.04it/s]


Test Loss: 1.6094 | Test Accuracy: 0.2000

===== Testing Best Model from Fold 3 =====


...

Test Loss: 2.6908 | Test Accuracy: 0.1053

===== Testing Best Model from Fold 4 =====


Evaluating: 100%|██████████| 200/200 [00:11<00:00, 17.86it/s]


Test Loss: 2.7081 | Test Accuracy: 0.0667

===== Testing Best Model from Fold 5 =====


Evaluating: 100%|██████████| 200/200 [00:11<00:00, 17.92it/s]

Test Loss: 2.6997 | Test Accuracy: 0.1017
Way 15 Average Test Accuracy: 9.05% ± 2.20% (1-sigma)
